# <center>Performance Optimization and Memory Efficiency

This phase teaches you how professionals work with datasets containing millions of rows.\
**The goal is:**\
Write Pandas code that is:\
✔ Fast\
✔ Memory-efficient\
✔ Production-ready\
✔ Scalable\
Most beginners only care about making code work.\
**Professionals care about:**\
How much RAM?\
How much time?\
Will this work on 50 million rows?

**Dataset**

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "ID" : np.arange(100000),
    "Department" : np.random.choice(["HR","IT","Finance"],100000),
    "Salary" : np.random.randint(40000,150000,100000)
})
df

,ID,Department,Salary
0,0,HR,46856
1,1,Finance,119044
2,2,HR,114548
3,3,Finance,108115
4,4,HR,87324
...,...,...,...
99995,99995,HR,90796
99996,99996,IT,56609
99997,99997,IT,66348
99998,99998,IT,139619


**1. Measuring Memory Usage**\
First rule : You cannot optimize what you don't measure.\
**memory_usage():**

In [2]:
df.memory_usage()

Index            132
ID            800000
Department    800000
Salary        400000
dtype: int64

**Total Memory:**

In [3]:
df.memory_usage().sum()

np.int64(2000132)

**Human Friendly in MBs**

In [4]:
df.memory_usage().sum() / 1024**2

np.float64(1.9074745178222656)

**2. Deep Memory Inspection**\
Object columns are tricky.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   ID          100000 non-null  int64 
 1   Department  100000 non-null  object
 2   Salary      100000 non-null  int32 
dtypes: int32(1), int64(1), object(1)
memory usage: 1.9+ MB


In [8]:
df.info(
    memory_usage = "deep"
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   ID          100000 non-null  int64 
 1   Department  100000 non-null  object
 2   Salary      100000 non-null  int32 
dtypes: int32(1), int64(1), object(1)
memory usage: 6.2 MB


In [9]:
df["Department"].info(
    memory_usage = "deep"
)

<class 'pandas.core.series.Series'>
RangeIndex: 100000 entries, 0 to 99999
Series name: Department
Non-Null Count   Dtype 
--------------   ----- 
100000 non-null  object
dtypes: object(1)
memory usage: 5.0 MB


**Why?**:\
Because object dtype stores pointers to Python objects.\
The actual strings are elsewhere in memory.

**3. astype()**\
One of the easiest optimizations.

**Common Optimizations**

a. Large Integers\
int64 --> int32

b. Small Integers\
int64 --> int16

c. Floating points\
float64-->float32

**Professional Rule:** Use the smallest datatype that safely holds your values

**4. Category Dtype**\
One of the biggest memory optimizations in Pandas 

In [13]:
df["Department"] = df[
    "Department"
].astype(
    "category"
)

In [14]:
df["Department"].dtype

CategoricalDtype(categories=['Finance', 'HR', 'IT'], ordered=False, categories_dtype=object)

In [16]:
df.memory_usage()

Index            132
ID            800000
Department    100132
Salary        400000
dtype: int64

**Before:**\
HR HR HR IT IT Finance\
Each string stored separately.

**After:**\
Categories:\
0 → Finance\
1 → HR\
2 → IT\

**Internally:**\
1 1 1 2 2 0\
Huge memory savings.

**5. Vectorization**

Bad:

In [20]:
for i in range(len(df)):

    df.loc[i,"Bonus"] = (
        df.loc[i,"Salary"]
        * 0.1
    )

This is slow.

Good:

In [22]:
df["Bonus"] = (
    df["Salary"]
    * 0.1
)

This is very fast

**6. iterrows() vs itertuples()**\
Sometimes loops are unavoidable\
**iterrows()**

In [23]:
for index,row in df.iterrows():

    print(
        row["Salary"]
    )

46856
119044
114548
108115
87324
148262
138500
71950
42116
45023
144020
123922
116680
63546
120549
129591
99670
149836
97924
140904
63981
135883
79085
91048
47990
142205
137420
105078
83488
104511
43388
61489
75993
120404
92930
65338
69134
42796
80330
143428
72546
141078
145627
117399
102641
139709
131581
59000
68673
102822
67417
52988
57244
136444
60443
135821
63228
48060
111498
81242
81094
143763
43290
141745
104171
42418
95443
119510
67820
46740
78379
121002
137111
131555
142889
142714
90053
96798
125792
57089
140762
123987
50605
72391
109047
96433
42338
132914
104438
81787
65602
134174
148444
79085
98995
148479
147812
104710
64255
120269
133899
65046
132200
53478
78278
131219
83642
56621
104808
140329
100015
144311
112346
81582
84701
143091
84077
49657
87069
64019
48846
141226
75497
106940
130863
119280
48111
139511
63752
86607
141885
128596
143836
55387
46455
68524
129974
109169
65439
129383
43100
40027
49968
108963
139447
93927
120882
67837
98045
57711
91813
142028
133109
128053


Very slow\
Because each row converts into series

**itertuples()**

In [24]:
for row in df.itertuples():

    print(
        row.Salary
    )

46856
119044
114548
108115
87324
148262
138500
71950
42116
45023
144020
123922
116680
63546
120549
129591
99670
149836
97924
140904
63981
135883
79085
91048
47990
142205
137420
105078
83488
104511
43388
61489
75993
120404
92930
65338
69134
42796
80330
143428
72546
141078
145627
117399
102641
139709
131581
59000
68673
102822
67417
52988
57244
136444
60443
135821
63228
48060
111498
81242
81094
143763
43290
141745
104171
42418
95443
119510
67820
46740
78379
121002
137111
131555
142889
142714
90053
96798
125792
57089
140762
123987
50605
72391
109047
96433
42338
132914
104438
81787
65602
134174
148444
79085
98995
148479
147812
104710
64255
120269
133899
65046
132200
53478
78278
131219
83642
56621
104808
140329
100015
144311
112346
81582
84701
143091
84077
49657
87069
64019
48846
141226
75497
106940
130863
119280
48111
139511
63752
86607
141885
128596
143836
55387
46455
68524
129974
109169
65439
129383
43100
40027
49968
108963
139447
93927
120882
67837
98045
57711
91813
142028
133109
128053


Way faster

**Professional Rule:** Never use iterrows() unless absolutely necessary.

**7. query()**\
Normal filtering:

In [25]:
df[
    (df["Salary"] > 80000)
    &
    (df["Department"]=="IT")
]

,ID,Department,Salary,Bonus
5,5,IT,148262,14826.2
6,6,IT,138500,13850.0
11,11,IT,123922,12392.2
16,16,IT,99670,9967.0
19,19,IT,140904,14090.4
...,...,...,...,...
99988,99988,IT,134227,13422.7
99989,99989,IT,142039,14203.9
99992,99992,IT,107625,10762.5
99994,99994,IT,114207,11420.7


Alternative:

In [26]:
df.query(
    "Salary > 80000 and Department=='IT'"
)

,ID,Department,Salary,Bonus
5,5,IT,148262,14826.2
6,6,IT,138500,13850.0
11,11,IT,123922,12392.2
16,16,IT,99670,9967.0
19,19,IT,140904,14090.4
...,...,...,...,...
99988,99988,IT,134227,13422.7
99989,99989,IT,142039,14203.9
99992,99992,IT,107625,10762.5
99994,99994,IT,114207,11420.7


**Advantages:**\
Cleaner\
More readable\
Sometimes faster

Industry teams often use:\
query()\
for complex conditions.

**8. eval()**\
Suppose:

In [27]:
df["Tax"] = (
    df["Salary"]*0.2
    +
    df["Bonus"]*0.1
)

Alternative:

In [28]:
df.eval(
    "Tax = Salary*0.2 + Bonus*0.1",
    inplace=True
)

**Benefits:**\
Less temporary memory\
Potentially faster\
Cleaner expressions\
Useful on huge datasets.

**9. Chunk Processing**\
Suppose:\
5 GB CSV\
RAM:\
8 GB\
You cannot load everything.

Instead of Entire file, you load in chunks and keep processing

This is how ETL pipelines work.

**10. Efficient CSV Reading**\
Read Specific Columns\
Bad:\
pd.read_csv(\
    "data.csv"\
)\
Better:\
pd.read_csv(\
    "data.csv",\
    usecols=["Salary","Department"]\
)\
**Specify Data Types**\
Bad:\
pd.read_csv(\
    "data.csv"\
)\
Better:\
pd.read_csv(\
    "data.csv",\
    dtype={\
        "Salary":"int32",\
        "Department":\
        "category"\
    }\
)\
This saves memory immediately.

**11. Copy vs View**\
This causes --> SettingWithCopyWarning

Bad:

In [29]:
high_salary = df[
    df["Salary"] > 80000
]

high_salary["Bonus"] = 10000

C:\Users\talpu\AppData\Local\Temp\ipykernel_23192\2111818719.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  high_salary["Bonus"] = 10000


Correct way:

In [31]:
high_salary = df[
    df["Salary"] > 80000
].copy()

Now, following line works safely:

In [32]:
high_salary["Bonus"] = 10000

**Professional Rule**: If you're creating a separate DataFrame, .copy() is your friend.

**12. apply() vs Vectorization**\
Bad:

In [33]:
df["Bonus"] = df[
    "Salary"
].apply(

    lambda x:
    x*0.1
)

Better:

In [34]:
df["Bonus"] = (
    df["Salary"]
    * 0.1
)

Vectorization almost always wins